In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.utils.dataframe_utils import create_dataframe, merge_dataset

df = create_dataframe("../data/train.csv", "../data/boneage-training-dataset")
df_val = create_dataframe("../data/validation.csv", "../data/boneage-validation-dataset")



In [ ]:
"""
#load of the segmented data
df_seg = create_dataframe("../data/train.csv", "../data/boneage-training-segmented")
df_val_seg = create_dataframe("../data/validation.csv", "../data/boneage-validation-dataset-segmented")

"""


In [ ]:
paths = df["path"].values
labels = df["boneage"].values

In [ ]:
"""
#create a new dataset with the segmented data

df = merge_dataset(df, df_segmented)
df_val = merge_dataset(df_val, df_val_segmented)
"""

In [ ]:

from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch


class BoneAgeDataset(Dataset):

    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform


    def __len__(self):
        return len(self.df)


    def __getitem__(self, idx):

        path = self.df.iloc[idx]["path"]
        label = self.df.iloc[idx]["boneage"]

        img = Image.open(path)

        if self.transform:
            img = self.transform(img)

        label = torch.tensor(label, dtype=torch.float32)

        return img, label

In [ ]:
dataset = BoneAgeDataset(
    df,
    transform=transform
)
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
dataset_val = BoneAgeDataset(
    df_val,
    transform=transform
)
loader_val = DataLoader(
    dataset_val,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [ ]:
images, labels = next(iter(loader))

print(images.shape)

In [ ]:
import torch
import torch.nn as nn


class BoneAgeCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            
            nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),


            nn.Conv2d(
                in_channels=64 if False else 32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),


            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1,1))
        )


        self.regressor = nn.Sequential(
            nn.Flatten(),

            nn.Linear(128,64),
            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(64,1)
        )


    def forward(self,x):

        x = self.features(x)
        x = self.regressor(x)

        return x

In [ ]:
model = BoneAgeCNN()

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

model = model.to(device)

cpu


In [ ]:
criterion = torch.nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [ ]:
epochs = 10

for epoch in range(epochs):

    model.train()

    running_loss = 0.0


    for images, labels in loader:

        # porto i dati sulla GPU
        images = images.to(device)
        labels = labels.to(device)


        # 1. azzero i gradienti vecchi
        optimizer.zero_grad()


        # 2. forward pass
        outputs = model(images)


        # 3. calcolo errore
        loss = criterion(outputs, labels)


        # 4. backpropagation
        loss.backward()


        # 5. aggiorno i pesi
        optimizer.step()


        running_loss += loss.item()


    print(
        f"Epoch {epoch+1}/{epochs} - loss:",
        running_loss / len(loader)
    )

In [ ]:
model.eval()

val_loss = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        val_loss += loss.item()


print(
    "Validation loss:",
    val_loss / len(val_loader)
)